## MIDI Tempo Acceleration and Report Generation

This section processes all generated MIDI files produced by the Data Availability-Based and Musical Diversity-Based Markov models.

Each MIDI file is loaded, its original tempo is detected, and the playback speed is increased using a predefined tempo multiplier. The symbolic note sequence, pitch order, and rhythmic structure remain unchanged; only the MIDI tempo metadata is modified.

The tempo-adjusted MIDI files are exported to a separate directory while preserving the original model-based folder structure. A CSV report is also created to document the source file, model group, original tempo, modified tempo, output file, and processing status.

**Output.** This cell reports the number of source MIDI files, successfully processed files, failed files, applied tempo multiplier, and generated report file.
data/generated_music/midi_tempo_modified/
midi_tempo_modified/
├── data_availability_based_markov_model/
└── musical_diversity_based_markov_model/

In [18]:
# ============================================================
# Accelerate generated MIDI files and save processing report
# ============================================================

from pathlib import Path

import pandas as pd
from mido import MetaMessage, MidiFile, bpm2tempo, tempo2bpm


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TEMPO_MULTIPLIER = 1.25
DEFAULT_BPM = 120


# ------------------------------------------------------------
# Locate the project root directory
# ------------------------------------------------------------

current_directory = Path.cwd()
project_directory = current_directory

while (
    project_directory.name != "TDC-Analysis-Book"
    and project_directory.parent != project_directory
):
    project_directory = project_directory.parent

if project_directory.name != "TDC-Analysis-Book":
    raise FileNotFoundError(
        "The TDC-Analysis-Book project directory could not be found."
    )


# ------------------------------------------------------------
# Define input and output directories
# ------------------------------------------------------------

INPUT_MIDI_DIRECTORY = (
    project_directory
    / "data"
    / "generated_music"
    / "midi"
)

OUTPUT_MIDI_DIRECTORY = (
    project_directory
    / "data"
    / "generated_music"
    / "midi_tempo_modified"
)

OUTPUT_MIDI_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Locate generated MIDI files
# ------------------------------------------------------------

midi_files = sorted(
    list(INPUT_MIDI_DIRECTORY.rglob("*.mid"))
    + list(INPUT_MIDI_DIRECTORY.rglob("*.midi"))
)

if not midi_files:
    raise FileNotFoundError(
        "No MIDI files were found in the generated music directory."
    )


# ------------------------------------------------------------
# Modify MIDI tempo
# ------------------------------------------------------------

processing_results = []

for midi_file in midi_files:

    try:
        midi = MidiFile(midi_file)

        relative_directory = midi_file.parent.relative_to(
            INPUT_MIDI_DIRECTORY
        )

        output_directory = (
            OUTPUT_MIDI_DIRECTORY
            / relative_directory
        )

        output_directory.mkdir(
            parents=True,
            exist_ok=True
        )

        output_file = (
            output_directory
            / f"{midi_file.stem}_tempo_{TEMPO_MULTIPLIER:.2f}x.mid"
        )

        original_tempos = []
        tempo_message_found = False

        for track in midi.tracks:
            for message in track:

                if message.type == "set_tempo":
                    tempo_message_found = True

                    original_bpm = tempo2bpm(message.tempo)
                    accelerated_bpm = (
                        original_bpm * TEMPO_MULTIPLIER
                    )

                    original_tempos.append(original_bpm)

                    message.tempo = bpm2tempo(
                        accelerated_bpm
                    )

        # Add a tempo message if the MIDI file has none
        if not tempo_message_found:
            accelerated_bpm = (
                DEFAULT_BPM * TEMPO_MULTIPLIER
            )

            midi.tracks[0].insert(
                0,
                MetaMessage(
                    "set_tempo",
                    tempo=bpm2tempo(accelerated_bpm),
                    time=0,
                ),
            )

            original_tempos.append(DEFAULT_BPM)

        midi.save(output_file)

        average_original_bpm = (
            sum(original_tempos)
            / len(original_tempos)
        )

        processing_results.append(
            {
                "source_file": midi_file.name,
                "model_group": relative_directory.as_posix(),
                "original_bpm": round(
                    average_original_bpm,
                    2,
                ),
                "tempo_multiplier": TEMPO_MULTIPLIER,
                "modified_bpm": round(
                    average_original_bpm
                    * TEMPO_MULTIPLIER,
                    2,
                ),
                "output_file": output_file.name,
                "status": "Successfully processed",
            }
        )

    except Exception as error:

        processing_results.append(
            {
                "source_file": midi_file.name,
                "model_group": midi_file.parent.name,
                "original_bpm": None,
                "tempo_multiplier": TEMPO_MULTIPLIER,
                "modified_bpm": None,
                "output_file": None,
                "status": f"Failed: {error}",
            }
        )


# ------------------------------------------------------------
# Create and save the processing report
# ------------------------------------------------------------

summary_df = pd.DataFrame(processing_results)

REPORT_FILE = (
    OUTPUT_MIDI_DIRECTORY
    / "midi_tempo_acceleration_report.csv"
)

summary_df.to_csv(
    REPORT_FILE,
    index=False,
)


# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

successful_files = (
    summary_df["status"]
    == "Successfully processed"
).sum()

failed_files = len(summary_df) - successful_files

print("=" * 60)
print("        MIDI Tempo Modification Completed")
print("=" * 60)
print(f"Source MIDI files : {len(midi_files)}")
print(f"Processed files   : {successful_files}")
print(f"Failed files      : {failed_files}")
print(f"Tempo multiplier  : {TEMPO_MULTIPLIER:.2f}x")
print(f"Report file       : {REPORT_FILE.name}")
print("Status            : Successfully completed")

summary_df

        MIDI Tempo Modification Completed
Source MIDI files : 8
Processed files   : 8
Failed files      : 0
Tempo multiplier  : 1.25x
Report file       : midi_tempo_acceleration_report.csv
Status            : Successfully completed


,source_file,model_group,original_bpm,tempo_multiplier,modified_bpm,output_file,status
0,AI_generated_data_availability_hicaz_duyek.mid,data_availability,120.0,1.25,150.0,AI_generated_data_availability_hicaz_duyek_tem...,Successfully processed
1,AI_generated_data_availability_nihavent_duyek.mid,data_availability,120.0,1.25,150.0,AI_generated_data_availability_nihavent_duyek_...,Successfully processed
2,AI_generated_data_availability_based_markov_mo...,data_availability_based_markov_model,120.0,1.25,150.0,AI_generated_data_availability_based_markov_mo...,Successfully processed
3,AI_generated_data_availability_based_markov_mo...,data_availability_based_markov_model,120.0,1.25,150.0,AI_generated_data_availability_based_markov_mo...,Successfully processed
4,AI_generated_musical_diversity_rast_duyek.mid,musical_diversity,120.0,1.25,150.0,AI_generated_musical_diversity_rast_duyek_temp...,Successfully processed
5,AI_generated_musical_diversity_based_markov_mo...,musical_diversity_based_markov_model,120.0,1.25,150.0,AI_generated_musical_diversity_based_markov_mo...,Successfully processed
6,AI_generated_musical_diversity_based_markov_mo...,musical_diversity_based_markov_model,120.0,1.25,150.0,AI_generated_musical_diversity_based_markov_mo...,Successfully processed
7,AI_generated_musical_diversity_based_markov_mo...,musical_diversity_based_markov_model,120.0,1.25,150.0,AI_generated_musical_diversity_based_markov_mo...,Successfully processed


## Next Chapter

The next chapter, **Related Datasets** presents the dataset structure and prepares them for the symbolic music processing workflows described in the following chapters.